In [11]:
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials

# Set your Spotify Developer credentials
CLIENT_ID = '744c0194864b419da63bde5738eab3f5'
CLIENT_SECRET = 'd44736a41bc2499980fc8db322e6f9f6'
PLAYLIST_ID = '5Tzd16z987zcxVm93pQQjU'  # Example: '37i9dQZF1DXcBWIGoYBM5M'

# Authenticate using client credentials flow
auth_manager = SpotifyClientCredentials(client_id=CLIENT_ID, client_secret=CLIENT_SECRET)
sp = spotipy.Spotify(auth_manager=auth_manager)

# Get all tracks from the playlist
def get_playlist_tracks(playlist_id):
    results = []
    offset = 0
    limit = 100

    while True:
        response = sp.playlist_items(playlist_id, offset=offset, limit=limit)
        items = response['items']
        if not items:
            break

        for item in items:
            track = item.get('track')
            if track:
                name = track.get('name')
                # artists = ', '.join([artist['name'] for artist in track['artists']])
                artist = track['artists'][0]['name']
                album_art = track['album']['images'][0]['url']
                results.append({
                    "name": name,
                    "artists": artist,
                    "album_art": album_art
                })

        offset += limit

    return results

# Example usage
tracks = get_playlist_tracks(PLAYLIST_ID)
tracks


[{'name': 'Reasons',
  'artists': 'Project 46',
  'album_art': 'https://i.scdn.co/image/ab67616d0000b273fde69fcb98bc47fb018860d3'},
 {'name': 'Surrender',
  'artists': 'Cash Cash',
  'album_art': 'https://i.scdn.co/image/ab67616d0000b27359793ced68eb51690ecd9f8f'},
 {'name': 'HELLO',
  'artists': 'Slevpy808',
  'album_art': 'https://i.scdn.co/image/ab67616d0000b2739dc42aa920708aec749b5e7d'},
 {'name': 'Mind Games',
  'artists': 'Alphabet Pony',
  'album_art': 'https://i.scdn.co/image/ab67616d0000b27363bfdaae334f9e4bd7db0f53'},
 {'name': 'Alone',
  'artists': 'Armin van Buuren',
  'album_art': 'https://i.scdn.co/image/ab67616d0000b273048012a99d7a62440392f529'},
 {'name': 'Wildfire - Radio Edit',
  'artists': 'Borgeous',
  'album_art': 'https://i.scdn.co/image/ab67616d0000b273de4c30446daffd2459add78f'},
 {'name': 'Angels x Demons - Radio Edit',
  'artists': 'Julian Jordan',
  'album_art': 'https://i.scdn.co/image/ab67616d0000b273af4d4ff225bb291a1d43f8c9'},
 {'name': 'Invincible',
  'artis

In [12]:
import torch
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import requests
from io import BytesIO

# Load CLIP model + processor
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# Device (GPU if available)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

# Store embeddings
clip_embeddings = []

for track in tracks:
    text = f"{track['name']} by {track['artists']}"
    image_url = track['album_art']

    # Load image
    response = requests.get(image_url)
    image = Image.open(BytesIO(response.content)).convert("RGB")

    # Prepare inputs
    inputs = processor(text=[text], images=[image], return_tensors="pt", padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Forward pass
    with torch.no_grad():
        outputs = model(**inputs)
        embedding = torch.cat([outputs.image_embeds, outputs.text_embeds], dim=1)  # concat image + text

    # Save embedding
    clip_embeddings.append({
        "track": track,
        "embedding": embedding.cpu().numpy()
    })

print(f"✅ Created {len(clip_embeddings)} combined CLIP embeddings (image + text)")


✅ Created 30 combined CLIP embeddings (image + text)


In [15]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Prompt for similarity search
prompt = "spinnin records"

# Encode prompt as text embedding
with torch.no_grad():
    text_inputs = processor(text=[prompt], return_tensors="pt", padding=True).to(device)
    text_embedding = model.get_text_features(**text_inputs)
    text_embedding = text_embedding.cpu().numpy()

# Extract image embeddings from previous step
image_embeddings = np.array([entry["embedding"][0][:512] for entry in clip_embeddings])  # first 512 = image
track_metadata = [entry["track"] for entry in clip_embeddings]

# Compute cosine similarity
similarities = cosine_similarity(text_embedding, image_embeddings)[0]

# Rank and display results
ranked_indices = np.argsort(similarities)[::-1]

print(f"\n🔍 Top matches for: '{prompt}'\n")
for idx in ranked_indices:
    track = track_metadata[idx]
    score = similarities[idx]
    print(f"{track['name']} by {track['artists']} — Similarity: {score:.4f}")



🔍 Top matches for: 'spinnin records'

In Your Arms - Radio Edit by Michael Woods — Similarity: 0.3200
Invincible by Borgeous — Similarity: 0.2981
Angels x Demons - Radio Edit by Julian Jordan — Similarity: 0.2958
Leaving You by Audien — Similarity: 0.2559
Nothing Inside by Sander van Doorn — Similarity: 0.2403
Colors - Audien Remix by Halsey — Similarity: 0.2331
Alone by Armin van Buuren — Similarity: 0.2299
Reasons by Project 46 — Similarity: 0.2249
HELLO by Slevpy808 — Similarity: 0.2215
Nothing Can Hold Us Down (feat. Haris) by Hardwell — Similarity: 0.2168
Steal You Away by Dash Berlin — Similarity: 0.2159
Something Better by Audien — Similarity: 0.2150
Lost by RSCL — Similarity: 0.2126
Runaway (U & I) by Galantis — Similarity: 0.2102
Chained To The Rhythm - Oliver Heldens Remix by Katy Perry — Similarity: 0.2074
DEAD IN THE DIRT by Holoblack — Similarity: 0.2050
All These Roads (feat. Zella Day and Sam Martin) by Sultan — Similarity: 0.2049
Love Again by Cedric Gervais — Similari

In [4]:
import requests as http
import urllib.parse
import time

for track in tracks:
    base_url = "https://musicbrainz.org/ws/2/recording"
    query = f'recording:"{track['name']}" AND artist:"{track['artists']}"'
    params = {
        'query': query,
        'fmt': 'json',
        'limit': 1
    }
    headers = {
        'User-Agent': 'Moodify/0.1.0 (a.williams.chase@gmail.com)'  # Use a real contact
    }
    url = f"{base_url}?{urllib.parse.urlencode(params)}"
    response = http.get(url, headers=headers)
    if response.status_code == 200:
        data = response.json()
        if data.get('recordings'):
            musicbrainz_recording = data['recordings'][0]
            track['recording_mbid'] = musicbrainz_recording['id']
            track['release_mbid'] = musicbrainz_recording['releases'][0]['id']
        else:
            print(data)
    else:
        print(response.status_code)
    time.sleep(1)
tracks

{'created': '2025-05-27T21:59:35.807Z', 'count': 0, 'offset': 0, 'recordings': []}
{'created': '2025-05-27T21:57:10.771Z', 'count': 0, 'offset': 0, 'recordings': []}
{'created': '2025-05-27T21:57:13.514Z', 'count': 0, 'offset': 0, 'recordings': []}
{'created': '2025-05-27T21:57:14.888Z', 'count': 0, 'offset': 0, 'recordings': []}
{'created': '2025-05-27T21:59:44.087Z', 'count': 0, 'offset': 0, 'recordings': []}
{'created': '2025-05-27T21:59:45.495Z', 'count': 0, 'offset': 0, 'recordings': []}
{'created': '2025-05-27T21:59:49.714Z', 'count': 0, 'offset': 0, 'recordings': []}
{'created': '2025-05-27T21:59:53.924Z', 'count': 0, 'offset': 0, 'recordings': []}
{'created': '2025-05-27T21:59:55.315Z', 'count': 0, 'offset': 0, 'recordings': []}
{'created': '2025-05-27T22:00:00.958Z', 'count': 0, 'offset': 0, 'recordings': []}
{'created': '2025-05-27T22:00:03.761Z', 'count': 0, 'offset': 0, 'recordings': []}
{'created': '2025-05-27T22:00:12.256Z', 'count': 0, 'offset': 0, 'recordings': []}


[{'name': 'Reasons',
  'artists': 'Project 46',
  'mbid': '551f9e97-0f2c-4f3e-868f-9d03dbb984ed'},
 {'name': 'Surrender',
  'artists': 'Cash Cash',
  'mbid': '1bc298ad-d313-4be0-9bb9-a88c35084ae5'},
 {'name': 'HELLO', 'artists': 'Slevpy808'},
 {'name': 'Mind Games', 'artists': 'Alphabet Pony'},
 {'name': 'Alone',
  'artists': 'Armin van Buuren',
  'mbid': '050bb2c4-2bfb-4e41-8cca-6d6fc0f662cd'},
 {'name': 'Wildfire - Radio Edit', 'artists': 'Borgeous'},
 {'name': 'Angels x Demons - Radio Edit', 'artists': 'Julian Jordan'},
 {'name': 'Invincible',
  'artists': 'Borgeous',
  'mbid': '5b9ba116-0b42-4daa-8da4-2a2f2160c7b8'},
 {'name': 'Take Me Home (feat. Bebe Rexha)', 'artists': 'Cash Cash'},
 {'name': 'City Of Dreams - Radio Edit', 'artists': 'Dirty South'},
 {'name': 'Leaving You',
  'artists': 'Audien',
  'mbid': '4bc9c96f-2cc0-4091-b059-a6a6b3a8e6b5'},
 {'name': 'Set Yourself Free',
  'artists': 'Tiësto',
  'mbid': '21d82389-47e8-4abc-a392-8ca644fa4074'},
 {'name': 'In Your Arms - Rad